<a href="https://colab.research.google.com/github/SerJes03/simulacion_modelo_marcov/blob/main/simulacionmarcov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# HMM INTERACTIVO - DIAGNÓSTICO MÉDICO
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import ipywidgets as widgets
from IPython.display import display

np.random.seed(42)

# ============================================================
# CLASE HMM
# ============================================================

class HMMMedico:

    def __init__(self):

        self.estados = [
            'Saludable',
            'Resfriado',
            'Gripe'
        ]

        self.observaciones = [
            'Temperatura Normal',
            'Fiebre Leve',
            'Fiebre Alta'
        ]

        # Distribución inicial
        self.pi = np.array([
            0.7,
            0.2,
            0.1
        ])

        # Matriz de transición
        self.A = np.array([
            [0.8, 0.15, 0.05],
            [0.4, 0.4, 0.2],
            [0.2, 0.3, 0.5]
        ])

        # Matriz de emisión
        self.B = np.array([
            [0.9, 0.08, 0.02],
            [0.3, 0.5, 0.2],
            [0.1, 0.3, 0.6]
        ])

    # ========================================================
    # SIMULACIÓN
    # ========================================================

    def simular(self, dias=20):

        estados_ocultos = []
        observaciones = []

        estado_actual = np.random.choice( #selecciona un estado aleatorio basado en probabilidades.
            len(self.estados),
            p=self.pi
        )

        for _ in range(dias): #ciclo ejecuta la simulación día por día- 20 veces

            estados_ocultos.append(estado_actual)
#Aquí el modelo genera el síntoma observable
            observacion = np.random.choice(
                len(self.observaciones),
                p=self.B[estado_actual]
            )

            observaciones.append(observacion) #almacena la temperatura generada.
#Aquí ocurre la transición de estados.
            estado_actual = np.random.choice(
                len(self.estados),
                p=self.A[estado_actual]
            )

        return estados_ocultos, observaciones

    # ========================================================
    # VITERBI
    # ========================================================

    def viterbi(self, observaciones):

        n_estados = len(self.estados)
        T = len(observaciones)

        delta = np.zeros((T, n_estados))
        psi = np.zeros((T, n_estados), dtype=int)

        # Inicialización recorre cada estado posible
        for s in range(n_estados):

            delta[0, s] = (
                self.pi[s] *
                self.B[s, observaciones[0]]
            )

        # Recursión
        for t in range(1, T):

            for s in range(n_estados):

                probabilidades = (
                    delta[t - 1] * #probabilidad de transición.
                    self.A[:, s] #de la matriz de transición.
                )

                psi[t, s] = np.argmax(probabilidades)
#calcula la mejor probabilidad acumulada
                delta[t, s] = (
                    np.max(probabilidades) *
                    self.B[s, observaciones[t]]
                )

        # Backtracking el estado más probable al final de la secuencia.
        estados = np.zeros(T, dtype=int)

        estados[T - 1] = np.argmax(delta[T - 1])

        for t in range(T - 2, -1, -1):

            estados[t] = psi[t + 1, estados[t + 1]]

        return estados, delta

    # ========================================================
    # DISTRIBUCIÓN ESTACIONARIA
    # ========================================================

    def distribucion_estacionaria(self):

        eigvals, eigvecs = np.linalg.eig(self.A.T)

        indice = np.argmin(np.abs(eigvals - 1))

        estacionaria = np.real(eigvecs[:, indice])

        estacionaria = estacionaria / np.sum(estacionaria)

        return estacionaria

# ============================================================
# CREAR MODELO
# ============================================================

modelo = HMMMedico()

# ============================================================
# FUNCIÓN PRINCIPAL
# ============================================================

def simulacion_interactiva(dias, simulaciones):

    print("=" * 60)
    print("SIMULACIÓN HMM - DIAGNÓSTICO MÉDICO")
    print("=" * 60)

    # --------------------------------------------------------
    # SIMULACIÓN
    # --------------------------------------------------------

    estados_reales, observaciones = modelo.simular(dias)

    estados_inferidos, delta = modelo.viterbi(observaciones)

    estados_reales_txt = [
        modelo.estados[i]
        for i in estados_reales
    ]

    estados_inferidos_txt = [
        modelo.estados[i]
        for i in estados_inferidos
    ]

    observaciones_txt = [
        modelo.observaciones[i]
        for i in observaciones
    ]

    # --------------------------------------------------------
    # TABLA
    # --------------------------------------------------------

    df = pd.DataFrame({

        'Día': np.arange(1, dias + 1),

        'Estado Real': estados_reales_txt,

        'Observación': observaciones_txt,

        'Estado Inferido': estados_inferidos_txt

    })

    print("\nTABLA DE RESULTADOS\n")

    display(df)

    # --------------------------------------------------------
    # MÉTRICAS
    # --------------------------------------------------------

    precision = np.mean(
        np.array(estados_reales) ==
        np.array(estados_inferidos)
    )

    conteo = Counter(estados_reales_txt)

    print("\nMÉTRICAS MÉDICAS\n")

    for estado, cantidad in conteo.items():

        print(f"{estado}: {cantidad} días")

    print(f"\nPrecisión diagnóstica: {precision:.2%}")

    # ========================================================
    # GRÁFICA 1 - REAL VS VITERBI
    # ========================================================

    fig, ax = plt.subplots(figsize=(15,6))

    dias_x = np.arange(1, dias + 1)

    # Estado real
    ax.plot(
        dias_x,
        estados_reales,
        marker='o',
        linewidth=3,
        linestyle='-',
        label='Estado Real'
    )

    # Estado inferido
    ax.plot(
        dias_x,
        estados_inferidos,
        marker='s',
        linewidth=3,
        linestyle='--',
        label='Estado Inferido (Viterbi)'
    )

    ax.set_yticks([0,1,2])

    ax.set_yticklabels([
        'Saludable',
        'Resfriado',
        'Gripe'
    ])

    ax.set_xlabel('Días')

    ax.set_ylabel('Estado Médico')

    ax.set_title(
        'Comparación Estado Real vs Diagnóstico Inferido',
        fontsize=16,
        fontweight='bold'
    )

    ax.grid(True, alpha=0.3)

    ax.legend()

    plt.show()

    # ========================================================
    # GRÁFICA 2 - TEMPERATURA OBSERVADA
    # ========================================================

    plt.figure(figsize=(15,5))

    plt.bar(
        dias_x,
        observaciones
    )

    plt.yticks(
        [0,1,2],
        ['Normal', 'Leve', 'Alta']
    )

    plt.title(
        'Temperatura Observada',
        fontsize=16,
        fontweight='bold'
    )

    plt.xlabel('Días')

    plt.ylabel('Nivel de Temperatura')

    plt.grid(axis='y', alpha=0.3)

    plt.show()

    # ========================================================
    # GRÁFICA 3 - CONFIANZA
    # ========================================================

    confianza = np.max(delta, axis=1)

    plt.figure(figsize=(15,5))

    plt.fill_between(
        dias_x,
        confianza,
        alpha=0.4
    )

    plt.plot(
        dias_x,
        confianza,
        linewidth=3
    )

    plt.title(
        'Confianza del Diagnóstico',
        fontsize=16,
        fontweight='bold'
    )

    plt.xlabel('Días')

    plt.ylabel('Probabilidad')

    plt.grid(True, alpha=0.3)

    plt.show()

    # ========================================================
    # GRÁFICA 4 - MÉTRICAS MÉDICAS
    # ========================================================

    plt.figure(figsize=(8,5))

    plt.bar(
        conteo.keys(),
        conteo.values()
    )

    plt.title(
        'Cantidad de Días por Estado Médico',
        fontsize=16,
        fontweight='bold'
    )

    plt.xlabel('Estado')

    plt.ylabel('Número de Días')

    plt.grid(axis='y', alpha=0.3)

    plt.show()

    # ========================================================
    # DISTRIBUCIÓN ESTACIONARIA
    # ========================================================

    dist_est = modelo.distribucion_estacionaria()

    print("\nDISTRIBUCIÓN ESTACIONARIA\n")

    for i, estado in enumerate(modelo.estados):

        print(
            f"{estado}: {dist_est[i]:.4f}"
        )

    # ========================================================
    # GRÁFICA 5 - CONVERGENCIA
    # ========================================================
    # diferentes cantidades de simulaciones para analizar convergencia

    lista_sim = [
        10,
        50,
        100,
        500,
        1000,
        simulaciones
    ]

    errores = []

    for n_sim in lista_sim:

        conteo_tmp = np.zeros(3)

        for _ in range(n_sim):

            estados_tmp, _ = modelo.simular(dias)

            for estado in estados_tmp:

                conteo_tmp[estado] += 1

        probs = conteo_tmp / np.sum(conteo_tmp)

        error = np.linalg.norm(
            probs - dist_est
        )

        errores.append(error)

    plt.figure(figsize=(10,5))

    plt.plot(
        lista_sim,
        errores,
        marker='o',
        linewidth=3
    )

    plt.xscale('log')

    plt.title(
        'Convergencia hacia la Distribución Estacionaria',
        fontsize=16,
        fontweight='bold'
    )

    plt.xlabel('Número de Simulaciones')

    plt.ylabel('Error')

    plt.grid(True, alpha=0.3)

    plt.show()

# ============================================================
# INTERFAZ INTERACTIVA
# ============================================================

dias_slider = widgets.IntSlider(
    value=20,
    min=10,
    max=100,
    step=10,
    description='Días'
)

sim_slider = widgets.IntSlider(
    value=5000,
    min=100,
    max=10000,
    step=100,
    description='Simulaciones'
)

boton = widgets.Button(
    description='Ejecutar Simulación',
    button_style='success',
    icon='play'
)

salida = widgets.Output()

# ============================================================
# FUNCIÓN BOTÓN
# ============================================================

def ejecutar_simulacion(b):

    salida.clear_output()

    with salida:

        simulacion_interactiva(
            dias_slider.value,
            sim_slider.value
        )

# ============================================================
# EVENTO BOTÓN
# ============================================================

boton.on_click(ejecutar_simulacion)

# ============================================================
# MOSTRAR INTERFAZ
# ============================================================

display(dias_slider)
display(sim_slider)
display(boton)
display(salida)